In [1]:
# !pip install -q datasets

In [2]:
!nvidia-smi

Sat May 18 12:25:51 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 530.30.02              Driver Version: 530.30.02    CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                  Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-80GB           Off| 00000000:07:00.0 Off |                    0 |
| N/A   34C    P0               69W / 400W|   1016MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [3]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [4]:
from datasets import load_dataset

dataset = load_dataset("TeeA/ChartQA")
dataset

/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id_image', 'image', 'table', 'chart_type', 'qa', 'vi_qa', 'vi_table'],
        num_rows: 18317
    })
    validation: Dataset({
        features: ['id_image', 'image', 'table', 'chart_type', 'qa', 'vi_qa', 'vi_table'],
        num_rows: 1056
    })
    test: Dataset({
        features: ['id_image', 'image', 'table', 'chart_type', 'qa', 'vi_qa', 'vi_table'],
        num_rows: 1509
    })
})

In [5]:
def helper(sample):
    sample['table'] = sample['table'].replace("<|>", " <|> ").replace("<&>", " <&> ")
    return sample
dataset = dataset.map(helper)

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [7]:
device = torch.device("cuda")

tokenizer_en2vi = AutoTokenizer.from_pretrained("vinai/vinai-translate-en2vi-v2", src_lang="en_XX")
model_en2vi = AutoModelForSeq2SeqLM.from_pretrained("vinai/vinai-translate-en2vi-v2").to(device)

In [8]:
def translate_en2vi(en_texts: list) -> str:
    input_ids = tokenizer_en2vi(en_texts, padding=True, return_tensors="pt").to(device)
    output_ids = model_en2vi.generate(
        **input_ids,
        decoder_start_token_id=tokenizer_en2vi.lang_code_to_id["vi_VN"],
        num_return_sequences=1,
        num_beams=5,
        early_stopping=True
    )
    vi_texts = tokenizer_en2vi.batch_decode(output_ids, skip_special_tokens=True)
    return vi_texts

In [9]:
translate_en2vi(dataset['train'][0]['table'].replace("<|>", " <|> ").replace("<&>", " <&> "))

["Đặc điểm <|> Tổng doanh thu trọn đời tính bằng triệu đô la Mỹ <&> Star Wars: Thần lực thức tỉnh <|> 936.66 <&> Avengers: Endgame* <|> 842.76 <&> Avatar <|> 760.51 <&> Black Panther <|> 700.06 <&> Avengers: Infinity War <|> 678.82 <&> Jurassic World <|> 652.27 <&> Marvel's The Avengers <|> 623.36 <&> Star Wars: Jedi cuối cùng <|> 620.18 <&> Incredibles 2 <|> 608.59 <&> Rogue One: A Star Wars Story <|> 532.18 <&> Beauty and the Beast (2017) <|> 504.01 <&> Finding Dory <|> 486.3 <&> Avengers: Age of Ultron <|> 459.01 <&> Captain Marvel* <|> 426.82 <&> Jurassic World: Fallen Kingdom <|> 416.77 <&> Toy Story 3 <|> 415.0 <&> Wonder Woman <|> 412.56 <&> Iron Man 3 <|> 409.01 <&> Captain America: Civil War <|> 408.08 <&> Jumanji: Welcome to the Jungle <|> 404.52"]

In [10]:
def translate_en2vi_batch(batch, stt):
    batch['vi_table'] = translate_en2vi(batch['table'])
    vi_qa = []
    for sample_qas in batch['qa']:
        vi_qa.append([])
        for qa in sample_qas:
            vi_qa[-1].append(
                {
                    'query': translate_en2vi(qa['query'])[0],
                    'label': translate_en2vi(qa['label'])[0]
                }
            )
    batch['vi_qa'] = vi_qa
    return batch

In [11]:
# translate_en2vi_batch(dataset['train'][:3])

In [ ]:
new_dataset = dataset.map(translate_en2vi_batch, batched=True, batch_size=10, with_indices=True)
new_dataset

Map:  69%|██████▉   | 12620/18317 [3:07:34<1:22:32,  1.15 examples/s]

In [ ]:
new_dataset.push_to_hub("TeeA/ChartQA", config_name="vin")

In [ ]:
!nvidia-smi

In [ ]:
!nvidia-smi

In [ ]:
!nvidia-smi